# Aspect-Based Sentiment Analysis (ABSA): Ingestion, Sentence Segmentation, Keyword Tagging, and Annotation Setup

In [3]:
!python -m spacy download en_core_web_sm

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


Using Python 3.13.12 environment at: C:\Users\Dilhara Jayawardhana\Projects\Dev\DS-Projects\letterboxed_absa\.venv
Checked 1 package in 19ms


In [4]:
import os
import re
import html
import json
import sqlite3
import ftfy
import numpy as np
import pandas as pd
import emoji
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import spacy

# Initialize spaCy with sentence boundary detection only for optimal performance
nlp = spacy.load("en_core_web_sm", disable=["tagger", "parser", "ner", "lemmatizer"])
nlp.enable_pipe("senter")

DB_PATH = "../data/raw/letterboxd.db"
RANDOM_STATE = 42

print("Dependencies and NLP pipeline loaded.")

Dependencies and NLP pipeline loaded.


In [5]:
def clean_for_transformers(text: str) -> str:
    """Removes noise and encoding errors while preserving NLP semantics."""
    if not isinstance(text, str):
        return ""
    
    # 1. Remove HTML/XML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # 2. Remove Markdown table artifacts (e.g., stray pipes)
    text = re.sub(r'^\||\|$', '', text.strip())
    
    # 3. Fix Mojibake encoding artifacts & decode HTML entities
    text = ftfy.fix_text(text)
    text = html.unescape(text)
    
    # 4. Remove Emojis
    text = emoji.replace_emoji(text, replace='')
    
    # 5. Standardize quotes/dashes (keeps contractions intact)
    unicode_replacements = {
        "’": "'", "‘": "'", "‚": "'", "‛": "'", "`": "'",
        "“": '"', "”": '"', "„": '"', "‟": '"',
        "—": " - ", "–": " - ", "―": " - ",
        "\xa0": " ",
        "\u200b": ""
    }
    for bad_char, good_char in unicode_replacements.items():
        text = text.replace(bad_char, good_char)
        
    # 6. Normalize excessive punctuation but keep the punctuation itself (!!! -> !, ??? -> ?)
    text = re.sub(r'(!){2,}', '!', text)
    text = re.sub(r'(\?){2,}', '?', text)
    
    # 7. Collapse whitespace and newlines into single spaces
    return re.sub(r'\s+', ' ', text).strip()

In [6]:
conn = sqlite3.connect(DB_PATH)

query = """
SELECT 
    r.review_id,
    r.film_id,
    f.film_name,
    CAST(f.release_year AS INTEGER) AS release_year,
    f.overall_rating AS film_overall_rating,
    r.star_rating,
    r.review_date,
    r.review_text,
    GROUP_CONCAT(DISTINCT g.genre_name) AS genres
FROM film_reviews r
JOIN films f ON r.film_id = f.film_id
LEFT JOIN film_genres g ON f.film_id = g.film_id
WHERE r.review_text IS NOT NULL 
  AND TRIM(r.review_text) != ''
GROUP BY r.review_id;
"""

df_reviews = pd.read_sql(query, conn)
conn.close()

# Clean raw metadata (film names may have encoding issues too)
df_reviews["film_name"] = df_reviews["film_name"].apply(clean_for_transformers)
df_reviews["star_rating_num"] = pd.to_numeric(df_reviews["star_rating"], errors="coerce")

print(f"Loaded {len(df_reviews):,} reviews across {df_reviews['film_id'].nunique():,} films.")

Loaded 175,396 reviews across 1,754 films.


In [7]:
sentences = []
batch_size = 2000

docs = nlp.pipe(df_reviews["review_text"], batch_size=batch_size, n_process=-1)

for doc, (_, row) in tqdm(zip(docs, df_reviews.iterrows()), total=len(df_reviews), desc="Segmenting & Filtering"):
    sent_idx = 0
    for sent in doc.sents:
        cleaned_sent = clean_for_transformers(sent.text)
        
        # Word count for quality filtering
        word_count = len(cleaned_sent.split())
        
        # Intelligent Quality Filtering (Keep >= 3 words)
        if word_count >= 3:
            sentences.append({
                "review_id": row["review_id"],
                "film_id": row["film_id"],
                "film_name": row["film_name"],
                "genres": row["genres"],
                "star_rating_num": row["star_rating_num"],
                "sentence_id": f"{row['review_id']}_{sent_idx}",
                "sentence_text": cleaned_sent,
                "is_truncated": cleaned_sent.endswith("...") # Flag truncated sentences
            })
            sent_idx += 1

df_sentences = pd.DataFrame(sentences)
print(f"Extracted {len(df_sentences):,} high-quality sentences.")

Segmenting & Filtering:   0%|          | 0/175396 [00:00<?, ?it/s]

Extracted 425,083 high-quality sentences.


In [10]:
import pyarrow

df_sentences.to_parquet("../data/processed/letterboxd_sentences.parquet",index=False)